# Convexity Adjustment

Plot Hull-White convexity diagnostics for the 1m and 3m SOFR futures strips.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.bootstrap import load_market_data
from src.daycount import yearfrac
from src.hw_model import U_j_const_sigma, convexity_1m

market = load_market_data(PROJECT_ROOT)
a = market.config.model.mean_reversion
sigma = market.config.model.sigma
valuation = market.config.market.valuation_date


In [ ]:
one_m = []
for quote in market.futures_1m:
    t0 = yearfrac(valuation, quote.start_date, 'ACT/365F')
    t1 = yearfrac(valuation, quote.end_date, 'ACT/365F')
    one_m.append((t1, 10_000 * convexity_1m(a, sigma, t0, t1)))

three_m = []
for quote in market.futures_3m:
    t0 = yearfrac(valuation, quote.start_date, 'ACT/365F')
    t1 = yearfrac(valuation, quote.end_date, 'ACT/365F')
    three_m.append((t1, 10_000 * U_j_const_sigma(a, sigma, t0, t1)))

plt.figure(figsize=(8, 4))
plt.plot([x for x, _ in one_m], [y for _, y in one_m], marker='o', label='1m convexity (bp)')
plt.plot([x for x, _ in three_m], [y for _, y in three_m], marker='s', label='3m U_j term (bp)')
plt.title('Convexity Diagnostics by Maturity')
plt.xlabel('Years from valuation')
plt.ylabel('Adjustment (bp)')
plt.legend()
plt.tight_layout()
plt.show()